# Entrega 6 — leitura dos resultados (T1–T4)

Este notebook **não calcula nada**: só lê `relatorios/<fonte>/e06.json`, gerado por
`python -m src.e06 --fonte real` (ou `--fonte amostra`). Todos os números do relatório saem de `src/`.

Salvo sem saídas no git: com a base real, as tabelas revelam volume. Rodar localmente.

In [ ]:
import json
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
FONTE = "real" if Path("../relatorios/real/e06.json").exists() else "amostra"
R = json.loads(Path(f"../relatorios/{FONTE}/e06.json").read_text(encoding="utf-8"))
print(FONTE, "| corte do treino:", R["corte_treino"], "| linhas treino/teste:", R["linhas_treino"], R["linhas_teste"])

## T1 — decomposição do ganho
Duas réguas: **B0 = média dos 3 últimos meses** (regra atual, devolutiva) e **B0 = P75 de 6 meses**
(baseline da E04/E05). B1 = s × B0 (s calibrado no treino); M = modelos de quantil (os mesmos nas duas).
Efeito quantil = B1 − B0; efeito modelagem = M − B1 (p.p. de pinball relativa; negativo = melhora).

In [ ]:
for k, r in R["T1"].items():
    print()
    print(k)
    display(pd.DataFrame(r["quadro"]).drop(columns="MAE").set_index("abordagem"))
    display(pd.DataFrame(r["modelos"]).T)
    for base, b in r["bases"].items():
        print(base, "| s =", b["s_cobertura_treino"], "| cobertura no treino B0 -> B1:",
              b["cobertura_treino_B0_%"], "->", b["cobertura_treino_B1_%"])
        display(pd.DataFrame(b["efeitos"]).T)

## T2 — defasamento (lag 1 × 4 × 5)
Compare a correlação com a **referência nula** (meses embaralhados no código), não com zero.

In [ ]:
for v in ["todos_os_meses", "sem_meses_com_ruptura"]:
    print(v); display(pd.DataFrame(R["T2"][v]).T)

## T3 — quadro de comparação e unidade de decisão (1 ciclo = 1 mês de cobertura)

In [ ]:
t3 = R["T3"]
display(pd.DataFrame(t3["quadro"]).set_index("abordagem"))
display(pd.DataFrame(t3["decisao"]).set_index(["abordagem", "versao"]))
display(pd.DataFrame(t3["ganho_gb"]))
print("teste de vazamento:", t3["teste_vazamento"])

## T4 — erros e equidade
Pinball relativa à média do próprio grupo; `parcela_custo_falta_%` pesa falta 0,40 e sobra 0,026.

In [ ]:
t4 = R["T4"]
for col, gs in t4["grupos"].items():
    linhas = {(g, a): {**v[a], "linhas": v["linhas"], "codigos": v["codigos"], "demanda_%": v["parcela_demanda_%"]}
              for g, v in gs.items() for a in ["B0_media3", "B0_p75_6", "M_gb_quantil"]}
    print(col); display(pd.DataFrame(linhas).T)
print("concentração:", t4["concentracao"]); print("erro de dado:", t4["erro_de_dado"])
print("fora do filtro incluir:", t4["fora_do_filtro_incluir"])